In [1]:
import pandas as pd

df_water = pd.read_csv(
    "final_datasets/berlin_water_quality_2022_2026.csv"
)

print("Shape:", df_water.shape)
print("\nColumns:")
print(df_water.columns.tolist())

df_water.head()

Shape: (298, 28)

Columns:
['date', 'bathing_water_quality', 'coliform_bacteria', 'cyanobacteria_chl_a', 'e_coli', 'intestinal_enterococci', 'water_visibility_depth', 'water_temperature', 'lake', 'water_temperature_raw', 'water_temperature_qualifier', 'water_temperature_numeric', 'temperature_zero_flag', 'coliform_bacteria_raw', 'coliform_bacteria_qualifier', 'coliform_bacteria_numeric', 'e_coli_raw', 'e_coli_qualifier', 'e_coli_numeric', 'intestinal_enterococci_raw', 'intestinal_enterococci_qualifier', 'intestinal_enterococci_numeric', 'cyanobacteria_chl_a_raw', 'cyanobacteria_chl_a_qualifier', 'cyanobacteria_chl_a_numeric', 'water_visibility_depth_raw', 'water_visibility_depth_qualifier', 'water_visibility_depth_numeric']


,date,bathing_water_quality,coliform_bacteria,cyanobacteria_chl_a,e_coli,intestinal_enterococci,water_visibility_depth,water_temperature,lake,water_temperature_raw,...,e_coli_numeric,intestinal_enterococci_raw,intestinal_enterococci_qualifier,intestinal_enterococci_numeric,cyanobacteria_chl_a_raw,cyanobacteria_chl_a_qualifier,cyanobacteria_chl_a_numeric,water_visibility_depth_raw,water_visibility_depth_qualifier,water_visibility_depth_numeric
0,2022-04-20,NaN,NaN,NaN,"< 15,00","< 15,00","0,00",NaN,Flughafensee,NaN,...,15.0,"< 15,00",<,15.0,NaN,NaN,NaN,"0,00",NaN,0.0
1,2022-05-17,NaN,NaN,NaN,"< 15,00","< 15,00","0,00",NaN,Flughafensee,NaN,...,15.0,"< 15,00",<,15.0,NaN,NaN,NaN,"0,00",NaN,0.0
2,2022-05-31,NaN,NaN,NaN,"< 15,00","15,00","> 1,00",NaN,Flughafensee,NaN,...,15.0,"15,00",NaN,15.0,NaN,NaN,NaN,"> 1,00",>,1.0
3,2022-06-13,NaN,NaN,NaN,"< 15,00","< 15,00","> 1,00",NaN,Flughafensee,NaN,...,15.0,"< 15,00",<,15.0,NaN,NaN,NaN,"> 1,00",>,1.0
4,2022-06-27,NaN,NaN,NaN,"15,00","15,00","> 1,00",NaN,Flughafensee,NaN,...,15.0,"15,00",NaN,15.0,NaN,NaN,NaN,"> 1,00",>,1.0


In [3]:
# Load the combined weather dataset

df_weather_data = pd.read_csv(
    "final_datasets/df_weather_combined_2022_2026.csv"
)

df_weather_data["date"] = pd.to_datetime(df_weather_data["date"])

print("Shape:", df_weather_data.shape)
print("\nColumns:")
print(df_weather_data.columns.tolist())

df_weather_data.head()

Shape: (14822, 13)

Columns:
['station_id', 'date', 'tavg', 'tmin', 'tmax', 'prcp', 'wspd', 'tsun', 'cloud_cover', 'pres_dwd', 'station', 'source', 'pres_meteostat']


,station_id,date,tavg,tmin,tmax,prcp,wspd,tsun,cloud_cover,pres_dwd,station,source,pres_meteostat
0,NaN,2022-01-01,11.2,9.5,11.7,1.7,16.4,NaN,NaN,NaN,Alexanderplatz,Meteostat,1021.3
1,NaN,2022-01-02,9.5,8.1,10.9,2.0,17.7,NaN,NaN,NaN,Alexanderplatz,Meteostat,1013.0
2,NaN,2022-01-03,8.7,7.0,10.0,4.0,20.2,NaN,NaN,NaN,Alexanderplatz,Meteostat,1006.5
3,NaN,2022-01-04,6.8,5.4,7.7,3.0,12.1,NaN,NaN,NaN,Alexanderplatz,Meteostat,996.8
4,NaN,2022-01-05,3.7,3.0,4.9,2.3,18.8,NaN,NaN,NaN,Alexanderplatz,Meteostat,997.8


In [4]:
# Combine DWD and Meteostat temperature measurements
# DWD takes priority; Meteostat fills missing values

tmax_sources = df_weather_data.pivot_table(
    index=["station", "date"],
    columns="source",
    values="tmax",
    aggfunc="first"
)

weather_tmax = pd.DataFrame(index=tmax_sources.index)

weather_tmax["tmax"] = (
    tmax_sources["DWD"].combine_first(tmax_sources["Meteostat"])
)

weather_tmax["tmax_source"] = "Missing"

weather_tmax.loc[
    tmax_sources["DWD"].notna(), "tmax_source"
] = "DWD"

weather_tmax.loc[
    tmax_sources["DWD"].isna() & tmax_sources["Meteostat"].notna(),
    "tmax_source"
] = "Meteostat"

weather_tmax = weather_tmax.reset_index()

print("Shape:", weather_tmax.shape)
print("\nMeasurement sources:")
print(weather_tmax["tmax_source"].value_counts())

weather_tmax.head()

Shape: (12252, 4)

Measurement sources:
tmax_source
DWD          8183
Meteostat    4069
Name: count, dtype: int64


,station,date,tmax,tmax_source
0,Alexanderplatz,2022-01-01,11.7,Meteostat
1,Alexanderplatz,2022-01-02,10.9,Meteostat
2,Alexanderplatz,2022-01-03,10.0,Meteostat
3,Alexanderplatz,2022-01-04,7.7,Meteostat
4,Alexanderplatz,2022-01-05,4.9,Meteostat


In [5]:
# Calculate daily average maximum temperature across available stations

daily_weather = (
    weather_tmax
    .groupby("date")
    .agg(
        mean_station_tmax=("tmax", "mean"),
        stations_available=("station", "nunique"),
        max_station_tmax=("tmax", "max")
    )
    .reset_index()
)

print("Shape:", daily_weather.shape)
print("\nDate range:")
print(daily_weather["date"].min(), "to", daily_weather["date"].max())

daily_weather.head()

Shape: (1764, 4)

Date range:
2022-01-01 00:00:00 to 2026-10-30 00:00:00


,date,mean_station_tmax,stations_available,max_station_tmax
0,2022-01-01,12.428571,7,13.0
1,2022-01-02,11.514286,7,11.9
2,2022-01-03,10.442857,7,10.8
3,2022-01-04,7.300000,7,8.2
4,2022-01-05,4.857143,7,5.1


In [7]:
# Convert water sampling dates to datetime

df_water["date"] = pd.to_datetime(df_water["date"])

print("Water date type:", df_water["date"].dtype)
print("Weather date type:", daily_weather["date"].dtype)

Water date type: datetime64[us]
Weather date type: datetime64[us]


In [8]:
# Match each water-quality sample with the weather on its sampling date

water_weather = df_water.merge(
    daily_weather,
    on="date",
    how="left",
    validate="many_to_one",
    indicator=True
)

print("Water samples:", len(water_weather))
print("\nWeather matching:")
print(water_weather["_merge"].value_counts())

print("\nUnmatched sampling dates:")
print(
    water_weather.loc[
        water_weather["_merge"] == "left_only", "date"
    ].drop_duplicates().sort_values().tolist()
)

# Remove merge indicator after checking
water_weather = water_weather.drop(columns="_merge")

water_weather.head()

Water samples: 298

Weather matching:
_merge
both          298
left_only       0
right_only      0
Name: count, dtype: int64

Unmatched sampling dates:
[]


,date,bathing_water_quality,coliform_bacteria,cyanobacteria_chl_a,e_coli,intestinal_enterococci,water_visibility_depth,water_temperature,lake,water_temperature_raw,...,intestinal_enterococci_numeric,cyanobacteria_chl_a_raw,cyanobacteria_chl_a_qualifier,cyanobacteria_chl_a_numeric,water_visibility_depth_raw,water_visibility_depth_qualifier,water_visibility_depth_numeric,mean_station_tmax,stations_available,max_station_tmax
0,2022-04-20,NaN,NaN,NaN,"< 15,00","< 15,00","0,00",NaN,Flughafensee,NaN,...,15.0,NaN,NaN,NaN,"0,00",NaN,0.0,13.957143,7,15.1
1,2022-05-17,NaN,NaN,NaN,"< 15,00","< 15,00","0,00",NaN,Flughafensee,NaN,...,15.0,NaN,NaN,NaN,"0,00",NaN,0.0,21.357143,7,23.1
2,2022-05-31,NaN,NaN,NaN,"< 15,00","15,00","> 1,00",NaN,Flughafensee,NaN,...,15.0,NaN,NaN,NaN,"> 1,00",>,1.0,19.757143,7,21.0
3,2022-06-13,NaN,NaN,NaN,"< 15,00","< 15,00","> 1,00",NaN,Flughafensee,NaN,...,15.0,NaN,NaN,NaN,"> 1,00",>,1.0,21.200000,6,22.0
4,2022-06-27,NaN,NaN,NaN,"15,00","15,00","> 1,00",NaN,Flughafensee,NaN,...,15.0,NaN,NaN,NaN,"> 1,00",>,1.0,34.971429,7,35.6


In [9]:
# Inspect measurement availability and qualifiers

indicators = {
    "E. coli": ("e_coli_numeric", "e_coli_qualifier"),
    "Intestinal enterococci": (
        "intestinal_enterococci_numeric",
        "intestinal_enterococci_qualifier"
    ),
    "Water temperature": (
        "water_temperature_numeric",
        "water_temperature_qualifier"
    ),
    "Cyanobacteria": (
        "cyanobacteria_chl_a_numeric",
        "cyanobacteria_chl_a_qualifier"
    )
}

for name, (value_col, qualifier_col) in indicators.items():

    print(f"\n{name}")
    print("Available numeric values:", water_weather[value_col].notna().sum())
    print("Qualifier distribution:")
    print(
        water_weather[qualifier_col]
        .value_counts(dropna=False)
    )


E. coli
Available numeric values: 298
Qualifier distribution:
e_coli_qualifier
<      189
NaN    109
Name: count, dtype: int64

Intestinal enterococci
Available numeric values: 298
Qualifier distribution:
intestinal_enterococci_qualifier
<      240
NaN     58
Name: count, dtype: int64

Water temperature
Available numeric values: 115
Qualifier distribution:
water_temperature_qualifier
NaN    298
Name: count, dtype: int64

Cyanobacteria
Available numeric values: 54
Qualifier distribution:
cyanobacteria_chl_a_qualifier
NaN    298
Name: count, dtype: int64


In [10]:
# Check water temperature coverage by lake and year

water_temp = water_weather[
    water_weather["water_temperature_numeric"].notna()
].copy()

print("Total temperature measurements:", len(water_temp))
print("\nFlagged zero measurements:")
print(water_temp["temperature_zero_flag"].value_counts(dropna=False))

print("\nMeasurements by lake:")
print(water_temp.groupby("lake").size().sort_values(ascending=False))

print("\nMeasurements by year:")
print(water_temp.groupby(water_temp["date"].dt.year).size())

Total temperature measurements: 115

Flagged zero measurements:
temperature_zero_flag
False    115
Name: count, dtype: int64

Measurements by lake:
lake
Müggelsee       20
Weißensee       20
Flughafensee    19
Orankesee       19
Tegeler See     19
Wannsee         18
dtype: int64

Measurements by year:
date
2025    57
2026    58
dtype: int64


In [11]:
# Summarise water temperature by lake

water_temp_summary = (
    water_temp.groupby("lake")["water_temperature_numeric"]
    .agg(["count", "mean", "median", "min", "max"])
    .round(2)
    .sort_values("mean", ascending=False)
)

display(water_temp_summary)

,count,mean,median,min,max
lake,,,,,
Weißensee,20,21.12,21.65,16.9,25.9
Orankesee,19,20.76,21.20,17.3,25.2
Flughafensee,19,20.57,21.00,16.0,25.3
Müggelsee,20,19.93,20.25,14.5,23.4
Wannsee,18,19.74,20.40,12.3,23.3
Tegeler See,19,19.46,19.40,13.1,24.0


In [12]:
# Inspect sampling dates and matched air temperatures by lake

sampling_summary = (
    water_temp.groupby("lake")
    .agg(
        first_sample=("date", "min"),
        last_sample=("date", "max"),
        sample_count=("date", "count"),
        mean_water_temp=("water_temperature_numeric", "mean"),
        mean_air_tmax=("mean_station_tmax", "mean"),
        min_air_tmax=("mean_station_tmax", "min"),
        max_air_tmax=("mean_station_tmax", "max")
    )
    .round(2)
    .sort_values("mean_water_temp", ascending=False)
)

display(sampling_summary)

C:\Users\lodim\AppData\Local\Temp\ipykernel_9772\4122471626.py:14: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  .round(2)


,first_sample,last_sample,sample_count,mean_water_temp,mean_air_tmax,min_air_tmax,max_air_tmax
lake,,,,,,,
Weißensee,2025-04-29,2026-09-09,20,21.12,22.83,12.80,32.24
Orankesee,2025-04-29,2026-09-09,19,20.76,22.04,12.80,31.26
Flughafensee,2025-04-29,2026-09-09,19,20.57,22.45,12.80,32.24
Müggelsee,2025-05-06,2026-09-08,20,19.93,22.65,15.11,30.27
Wannsee,2025-04-29,2026-09-15,18,19.74,23.17,15.20,31.84
Tegeler See,2025-04-28,2026-09-14,19,19.46,22.00,15.01,29.60
